# **USD/IDR (IDR=X)**

In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo

#supaya row muncul semua

#Ambil data
start_date = "2025-08-01"
end_date = "2025-10-28"     #end date dibuat H+1 dari tanggal target

ticker = yf.Ticker("IDR=X")
df = ticker.history(start=start_date, end=end_date, interval="1d")

# Urutkan dari tanggal terbaru ke terlama dan normalisasi
df = df.sort_index(ascending=False)
df.index = df.index.tz_localize(None)
df.index = df.index.date
kolom_harga = [c for c in ["Open", "High", "Low", "Close"] if c in df.columns]
df[kolom_harga] = df[kolom_harga].round(4)


#save ke excel
wib = ZoneInfo("Asia/Jakarta")
timestamp = datetime.now(tz=wib).strftime("%Y%m%d-%H%M%S")
file_name = f"[{timestamp}] IDR X Yahoo Finance.xlsx"

df.to_excel(file_name, sheet_name="Data_Harian")
print(f"✅ Data berhasil disimpan ke '{file_name}'")
print(df.head())

✅ Data berhasil disimpan ke '[20251028-102305] IDR X Yahoo Finance.xlsx'
                  Open        High         Low       Close  Volume  Dividends  \
2025-10-27  16597.8008  16626.0000  16588.0000  16597.8008       0        0.0   
2025-10-24  16593.8008  16638.0000  16449.5996  16593.8008       0        0.0   
2025-10-23  16596.5996  16650.8008  16590.0000  16596.5996       0        0.0   
2025-10-22  16598.9004  16624.3008  16570.0000  16598.9004       0        0.0   
2025-10-21  16542.0000  16609.8008  16538.6992  16542.0000       0        0.0   

            Stock Splits  
2025-10-27           0.0  
2025-10-24           0.0  
2025-10-23           0.0  
2025-10-22           0.0  
2025-10-21           0.0  


# **Ringkasan Index PT BEI**

In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.firefox import GeckoDriverManager
from datetime import datetime, timedelta
import pandas as pd
import re


#Inisiasi driver
def get_firefox_options():
    firefox_options = Options()
    firefox_options.add_argument("--headless")
    firefox_options.add_argument("--no-sandbox")
    firefox_options.add_argument("--disable-gpu")
    firefox_options.set_preference("intl.accept_languages", "id")
    return firefox_options


def init_driver():
    firefox_options = get_firefox_options()  # Get Firefox options
    profile = webdriver.FirefoxProfile()
    profile.set_preference("browser.cache.disk.enable", False)
    profile.set_preference("browser.cache.memory.enable", False)
    profile.set_preference("browser.cache.offline.enable", False)
    firefox_options.profile = profile
    driver = webdriver.Firefox(service=Service(GeckoDriverManager().install()), options=firefox_options)        
    driver.set_window_size(1920, 1080)
    return driver

# Fungsi bantu untuk parsing teks menjadi list of dict
def parse_json_text(json_text):
    records = re.split(r'\nNo \d+\n', json_text)[1:]  # skip bagian sebelum No 1
    data_list = []

    for rec in records:
        lines = rec.split('\n')
        record = {}
        for line in lines:
            if ' ' in line:
                key, value = line.split(' ', 1)
                value = value.replace("JS:", "")
                try:
                    value = float(value)
                except:
                    value = value.strip('"')  # bersihkan tanda kutip jika string
                record[key] = value
        data_list.append(record)
    return data_list

In [3]:
# --- Input awal ---
start_date_str = "2025-10-01"
end_date_str = "2025-10-27"
filter_index_codes = ["IDX30"]  # bisa lebih dari 1 kode ["A","B","C",...]

# Parsing tanggal
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
end_date = datetime.strptime(end_date_str, "%Y-%m-%d")

# --- Inisialisasi driver ---
driver = init_driver()
all_data = []

try:
    current_date = start_date
    current_tab = None

    while current_date <= end_date:
        date_str_url = current_date.strftime('%Y%m%d')
        url = f"https://idx.co.id/primary/TradingSummary/GetIndexSummary?date={date_str_url}&length=9999&start=0"

        if current_tab is None:
            # Tab pertama
            driver.get(url)
            current_tab = driver.current_window_handle
        else:
            # Tab baru
            driver.execute_script("window.open('');")
            new_tab = driver.window_handles[-1]
            driver.switch_to.window(new_tab)
            driver.get(url)
            # Tutup tab lama
            driver.switch_to.window(current_tab)
            driver.close()
            # Switch ke tab baru
            driver.switch_to.window(new_tab)
            current_tab = new_tab

        # Ambil data
        json_text = driver.find_element(By.TAG_NAME, "body").text
        if json_text:
            print(f"Get Data for {current_date.strftime('%Y-%m-%d')} ... Success")
            records = parse_json_text(json_text)
            if records:
                for r in records:
                    r['Date'] = current_date.strftime('%Y-%m-%d')
                # Filter IndexCode
                records = [r for r in records if r['IndexCode'].upper() in filter_index_codes]
                all_data.extend(records)
            else:
                print(f"No records for {current_date.strftime('%Y-%m-%d')}")
        else:
            print(f"Failed to get data for {current_date.strftime('%Y-%m-%d')}")

        current_date += timedelta(days=1)

finally:
    driver.quit()

# --- Simpan ke Excel ---
#save ke excel
wib = ZoneInfo("Asia/Jakarta")
timestamp = datetime.now(tz=wib).strftime("%Y%m%d-%H%M%S")
file_name = f"[{timestamp}] {filter_index_codes} - IDX BEI.xlsx"


if all_data:
    df = pd.DataFrame(all_data)
    print("\n")
    df.to_excel(file_name, sheet_name="Data_Harian")
    print(f"✅ Data berhasil disimpan ke '{file_name}' dengan {len(all_data)} record")
    print("\n")
    print(df.head())

else:
    print("Tidak ada data yang sesuai filter.")


Get Data for 2025-10-01 ... Success
Get Data for 2025-10-02 ... Success
Get Data for 2025-10-03 ... Success
Get Data for 2025-10-04 ... Success
No records for 2025-10-04
Get Data for 2025-10-05 ... Success
No records for 2025-10-05
Get Data for 2025-10-06 ... Success
Get Data for 2025-10-07 ... Success
Get Data for 2025-10-08 ... Success
Get Data for 2025-10-09 ... Success
Get Data for 2025-10-10 ... Success
Get Data for 2025-10-11 ... Success
No records for 2025-10-11
Get Data for 2025-10-12 ... Success
No records for 2025-10-12
Get Data for 2025-10-13 ... Success
Get Data for 2025-10-14 ... Success
Get Data for 2025-10-15 ... Success
Get Data for 2025-10-16 ... Success
Get Data for 2025-10-17 ... Success
Get Data for 2025-10-18 ... Success
No records for 2025-10-18
Get Data for 2025-10-19 ... Success
No records for 2025-10-19
Get Data for 2025-10-20 ... Success
Get Data for 2025-10-21 ... Success
Get Data for 2025-10-22 ... Success
Get Data for 2025-10-23 ... Success
Get Data for 202

AttributeError: 'function' object has no attribute 'text'